In [4]:
import numpy as np
from scipy.optimize import minimize

# Константы из задания: скорость звука и частота дискретизации
SOUND_SPEED = 1125  # скорость звука (фут/с)
SAMPLE_RATE = 100000  # частота дискретизации (Гц)

# Координаты динамиков в углах комнаты (как на схеме из лабы)
SPEAKER_POSITIONS = np.array([
    [0, 0, 10],
    [20, 0, 10],
    [20, 20, 10],
    [0, 20, 10]
])

def compute_time_delay(received_signal, reference_signal):
    """
    Вычисление временной задержки между сигналами
    с использованием взаимной корреляции
    """
    # Взаимная корреляция показывает, при каком сдвиге T сигналы лучше всего совпадают
    correlation = np.correlate(received_signal, reference_signal)
    
    # Индекс пика корреляции соответствует искомой задержке
    peak_index = np.argmax(correlation)
    
    # Переводим индекс в секунды: делим на количество отсчётов в секунду
    return peak_index / SAMPLE_RATE

def compute_distances_from_delays(delays):
    """
    Перевод временных задержек в расстояния
    """
    # Используем формулу расстояния d = c * T из условия задачи
    return [delay * SOUND_SPEED for delay in delays]

def residuals_squared(point, measured_distances):
    """
    Функция невязок для оптимизации методом наименьших квадратов
    point: предполагаемые координаты микрофона [x, y, z]
    measured_distances: измеренные расстояния до громкоговорителей
    """
    x, y, z = point
    errors = []
    
    for idx, (sx, sy, sz) in enumerate(SPEAKER_POSITIONS):
        # Считаем геометрическое расстояние от предполагаемой точки до i-го динамика
        dx = x - sx
        dy = y - sy
        dz = z - sz
        calculated_distance = np.sqrt(dx*dx + dy*dy + dz*dz)
        
        # Невязка: разница между расчетным евклидовым расстоянием и измеренным по задержке
        errors.append(calculated_distance - measured_distances[idx])
    
    # Возвращаем сумму квадратов невязок. Это целевая функция, которую оптимизатор будет минимизировать
    return np.sum(np.square(errors))

# Загрузка данных
transmitter_signals = np.loadtxt("data_files/Transmitter.txt")  # сигналы с передатчиков (4xN)
receiver_signal = np.loadtxt("data_files/Receiver.txt")        # сигнал с приемника (1xN)

# Для каждого динамика находим задержку распространения звука до микрофона
time_delays = [
    compute_time_delay(receiver_signal, transmitter_signals[i])
    for i in range(len(SPEAKER_POSITIONS))
]

# Переводим все найденные задержки в физические расстояния
measured_ranges = compute_distances_from_delays(time_delays)

# Начальное приближение, с которого численный алгоритм начнёт поиск решения
initial_estimate = [1.0, 1.0, 1.0]

# Так как уравнений 4, а неизвестных координат 3, система переопределённая.
# Решаем нелинейным МНК: подбираем такие [x,y,z], чтобы сумма квадратов разниц между геометрией и замерами была минимальной
optimization_result = minimize(
    residuals_squared,
    initial_estimate,
    args=(measured_ranges,),
    method='L-BFGS-B'
)

# Забираем найденные координаты из результата оптимизации
x_coord, y_coord, z_coord = optimization_result.x

print(f"Локализация микрофона завершена")
print(f"Координаты: X = {x_coord:.2f}, Y = {y_coord:.2f}, Z = {z_coord:.2f}")
print(f"Невязка: {optimization_result.fun:.6f}")

Локализация микрофона завершена
Координаты: X = 8.00, Y = 4.60, Z = 5.76
Невязка: 0.000034
